In [ ]:
import pandas as pd
import numpy as np
import os
import re

arquivos_csv = [
    '../Dados/PNAD_COVID_052020.csv',
    '../Dados/PNAD_COVID_062020.csv',
    '../Dados/PNAD_COVID_072020.csv',
    '../Dados/PNAD_COVID_082020.csv',
    '../Dados/PNAD_COVID_092020.csv',
    '../Dados/PNAD_COVID_102020.csv',
    '../Dados/PNAD_COVID_112020.csv'
]

In [ ]:
def processar(df_original):

    df = df_original.copy()
    colunas_especificas = [
        'A004', 'A005',
        'B0011','B0012','B0013','B0015','B0018','B00111','B00112', 
        'B0014','B0016','B0017','B0019','B00110','B00113',
        'B002', 
        'B0031','B0032','B0033','B0034','B0035','B0036','B0037',
        'B0041','B0042','B0043','B0044','B0045','B0046',
        'B009B','B009D','B009F',
        'B0101','B0102','B0103','B0104','B0105','B0106',
        'B011',
        'C001','C002',
        'C01012','C01022','C011A12','C011A22',
        'C012','C013',
        'D0013','D0023','D0033','D0043','D0053','D0063','D0073',
        'E001',
        'F001',
        'F002A1','F002A2','F002A3','F002A4','F002A5'
    ]

    colunas_vdf = [c for c in colunas_especificas if c in df.columns]

    padrao_vdf = re.compile(r'^(V|D|F|A|C|B|E)[0-9A-Z]')

    todas_vdf = [c for c in df.columns if padrao_vdf.match(c) and c not in ['Ano','CAPITAL']]

    colunas_para_remover = [c for c in todas_vdf if c not in colunas_vdf]

    df = df.drop(columns=colunas_para_remover)

    anos = df["Ano"].unique()
    for a in anos:
        df[f"Ano_{int(a)}"] = (df["Ano"] == a).astype(int)

    df = df.drop(columns=["Ano"])

    colunas_outras = [
        c for c in df.columns
        if (not padrao_vdf.match(c) or c in ['CAPITAL'])
        and c not in colunas_vdf
    ]

    id_vars = colunas_outras + [c for c in df.columns if c.startswith("Ano_")]

    df_transformado = df.copy()

    def calc_grupo(df, cols):
        if not cols:
            return 2
        return np.where(df[cols].isin([1,1.0,'1']).any(axis=1), 1, 2)

    def calc_soma(df, cols):
        return df[cols].fillna(0).sum(axis=1)

    grupos = {
        "B1_Sintomas_Principais": ['B0011','B0012','B0013','B0015','B0018','B00111'],
        "B1_Outros_Sintomas": ['B0014','B0016','B0017','B0019','B00110','B00112','B00113'],
        "B3_Isolamento": ['B0031'],
        "B3_Buscou_Orientacao": ['B0032','B0034','B0035','B0036'],
        "B3_Automedicacao": ['B0033'],
        "B3_Outra": ['B0037'],
        "B4_Publico": ['B0041','B0042','B0043'],
        "B4_Privado": ['B0044','B0045','B0046'],
        "B9": ['B009B','B009D','B009F'],
        "B10": ['B0101','B0102','B0103','B0104','B0105','B0106'],
        "C10": ['C01012','C01022'],
        "C11": ['C011A12','C011A22'],
        "D1": ['D0013','D0023','D0033','D0043','D0053','D0063','D0073'],
        "F2A": ['F002A1','F002A2','F002A3','F002A4','F002A5']
    }

    for nome, cols in grupos.items():
        cols_validas = [c for c in cols if c in df_transformado.columns]
        if nome in ["C10", "C11", "D1"]:
            df_transformado[nome] = calc_soma(df_transformado, cols_validas)
        else:
            df_transformado[nome] = calc_grupo(df_transformado, cols_validas)

    colunas_grupos_originais = sum(grupos.values(), [])
    colunas_drop = [c for c in colunas_grupos_originais if c in df_transformado.columns]

    df_transformado = df_transformado.drop(columns=colunas_drop)

    novas_cols = [c for c in grupos.keys() if c in df_transformado.columns]

    df_final = df_transformado.copy()
    # pd.melt(
    #     df_transformado,
    #     id_vars=[c for c in df_transformado.columns if c not in todas_value_vars],
    #     value_vars=todas_value_vars,
    #     var_name='Variavel',
    #     value_name='Valor'
    # )

    return df_final


In [8]:
def processar_todos_arquivos(arquivos_csv):

    for arquivo in arquivos_csv:

        print(f"\n🔹 Processando arquivo: {arquivo}")

        if not os.path.exists(arquivo):
            print("❌ Arquivo não encontrado!")
            continue

        df = pd.read_csv(arquivo, low_memory=False)

        df_final = processar(df)

        nome_saida = os.path.basename(arquivo).replace(".csv", "_transformado.csv")

        df_final.to_csv(nome_saida, index=False, encoding="utf-8")

        print(f"✅ Arquivo salvo: {nome_saida}")


In [11]:
processar_todos_arquivos(arquivos_csv)


🔹 Processando arquivo: ../Dados/PNAD_COVID_052020.csv
✅ Arquivo salvo: PNAD_COVID_052020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_062020.csv
✅ Arquivo salvo: PNAD_COVID_062020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_072020.csv
✅ Arquivo salvo: PNAD_COVID_072020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_082020.csv
✅ Arquivo salvo: PNAD_COVID_082020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_092020.csv
✅ Arquivo salvo: PNAD_COVID_092020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_102020.csv
✅ Arquivo salvo: PNAD_COVID_102020_transformado.csv

🔹 Processando arquivo: ../Dados/PNAD_COVID_112020.csv
✅ Arquivo salvo: PNAD_COVID_112020_transformado.csv
